<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 6 - Your Final Fund (DRAFT v2)

In Studio 5 you built a fund with Ms. Womack's model and put it on the leaderboard. Tonight you build your **final
fund**: you choose the stocks, you choose the settings, the model sets the shares, and the fund runs until the market
closes on **Thursday, Dec 3**.

1. **Part 1 - Where I stand.** A quick look at your Studio 5 fund and the class.
2. **Part 2 - The quilt.** The 100 biggest companies in the S&P 500, over three stretches of time.
3. **Part 3 - My picks.** Choose 10 to 25 stocks and say what your strategy is.
4. **Part 4 - Solve.** Ms. Womack's model sets the shares. Change your picks or settings and solve again.
5. **Part 5 - The news (optional).** One file with the headlines for your fund, to read with Claude.
6. **Part 6 - Name it and send it.** Your fund goes on the leaderboard as **FINAL**.

**The rules for a final fund:** hold **at least 10 stocks**, and put **at most 25%** in any one.

The notebook comes with Dr. Dave's example picks, so it runs from top to bottom as it is: Runtime, Run all. Then make
it yours. You can come back and send a new version as often as you like until Dec 3.

## Setup

Pyomo, the two solvers (ipopt and Bonmin) and yfinance, as in Studio 5. About a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt and Bonmin)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
import json
import html
import requests
import yfinance as yf
import xml.etree.ElementTree as ET
from pyomo.environ import *
from IPython.display import display
import matplotlib.dates as mdates

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solvers
ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

## The data

Three things: the list of companies, the class's funds, and five years of daily prices.

**The 100 biggest companies** in the S&P 500 are tonight's menu. `sp500` has every company's name, sector and size.

In [ ]:
# every company in the S&P 500: its name, its sector and its size (market value, in billions of dollars)
sp500 = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/sp500_2025-12-31.csv').set_index('ticker')
sp500['sector'] = sp500['sector'].replace('Information Technology', 'Technology')

# the 100 biggest
biggest = list(sp500.sort_values('market_value_billions', ascending=False).head(100).index)

# the class: every student fund by its leaderboard name, and the class crowd fund (the average of them)
crowd_file = requests.get('https://storage.googleapis.com/opim5641-leaderboard-f26/crowd.json', timeout=30).json()
class_funds = crowd_file['named_funds']
crowd_mix = crowd_file['crowd']['weights']
class_start = pd.Timestamp(crowd_file['crowd']['start'])          # Nov 5: the day the class funds went live

print("The menu:", len(biggest), "companies, from", sp500.loc[biggest[0], 'company'], "to", sp500.loc[biggest[-1], 'company'])
print("The class:", len(class_funds), "student funds, live since", class_start.strftime('%b %d'))

**Prices.** One download from Yahoo Finance: daily closing prices since September 2021 for the 100 companies,
every stock the class holds, and the S&P 500 (`SPY`). If Yahoo does not answer, the cell uses a copy saved for class.

In [ ]:
# every ticker we need, once each
tickers = ['SPY']
for t in biggest + list(crowd_mix):
    if t not in tickers:
        tickers.append(t)

# ask Yahoo
try:
    prices = yf.download(tickers, start='2021-09-01', interval='1d', auto_adjust=True, progress=False)['Close']
except Exception as problem:
    prices = pd.DataFrame()

# the backup: a copy of the same prices saved for class
class_copy = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/daily_prices.csv', index_col=0, parse_dates=True)
if 'SPY' not in prices.columns or prices['SPY'].notna().sum() < 200:
    prices = class_copy.copy()
    print("Yahoo did not answer, so the prices come from the class copy.")
else:
    print("Yahoo answered.")

# a ticker Yahoo skipped: take it from the class copy
for t in tickers:
    if t not in prices.columns or prices[t].notna().sum() == 0:
        prices[t] = class_copy[t].reindex(prices.index)

# days in order, a missing day filled with the day before, and no unfinished day (before 4:30 PM Eastern, drop today)
prices = prices.sort_index().ffill()
now = pd.Timestamp.now(tz='America/New_York')
if now.hour * 60 + now.minute < 16 * 60 + 30:
    prices = prices[prices.index < pd.Timestamp(now.date())]
last_close = prices.index[-1]

print(prices.shape[1], "tickers,", prices.shape[0], "days, through the close of", last_close.strftime('%a %b %d, %Y'))

**`hold`** is the one helper we use all night: put \$10,000 into a mix of stocks on one day, leave it alone, and
return what it is worth at every close after that. (Studio 5, Part 4c.)

In [ ]:
def hold(mix, day):
    # $10,000 split by `mix` (stock -> share) at the close of `day`, then held
    window = prices.loc[day:]
    line = 0
    for t in mix:
        line = line + 10000 * mix[t] * window[t] / window[t].iloc[0]
    return line


spy_line = hold({'SPY': 1.0}, class_start)
crowd_line = hold(crowd_mix, class_start)
print("$10,000 since %s: the S&P 500 is at $%.0f, the class crowd fund is at $%.0f." % (
    class_start.strftime('%b %d'), spy_line.iloc[-1], crowd_line.iloc[-1]))

# Part 1 - Where I stand

### My Studio 5 fund

Paste your **record link**: in HuskyCT, open Gradebook, click **Studio 5 credit**, and copy the link in the feedback.
The link below is Dr. Dave's example fund. No Studio 5 fund? Make it `''` and go on to Part 2.

In [ ]:
record_link = 'https://storage.googleapis.com/opim5641-leaderboard-f26/records/8000437f223b5f668e7dee73.json'

In [ ]:
if record_link.strip() == '':
    print("No Studio 5 fund loaded. Go on to Part 2.")
else:
    my_record = requests.get(record_link.strip(), timeout=30).json()

    # my fund and its no-rules twin at every close, straight from the leaderboard
    days = pd.to_datetime(my_record['values']['dates'])
    my_line = pd.Series(my_record['values']['with_rules'], index=days)
    twin_line = pd.Series(my_record['values']['no_rules'], index=days)

    # the S&P 500 over the same days
    market_line = hold({'SPY': 1.0}, days[0])

    fig, panel = subplots(figsize=(11, 4.5))
    panel.plot(market_line.index, market_line.values, color='black', linewidth=2, label='S&P 500')
    panel.plot(twin_line.index, twin_line.values, color='gray', linewidth=1.5, linestyle='--', label='its no-rules twin')
    panel.plot(my_line.index, my_line.values, color='tab:green', linewidth=3, marker='o', markersize=4, label=my_record['nickname'])
    panel.axhline(10000, color='lightgray', linewidth=1)
    panel.set_title('$10,000 since the %s close' % days[0].strftime('%b %d'))
    panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    panel.legend()
    show()

    print("%s: $10,000 became $%.0f." % (my_record['nickname'], my_line.iloc[-1]))
    print("Its no-rules twin (the plain Ms. Womack model, same stocks, no rules): $%.0f." % twin_line.iloc[-1])
    print("The S&P 500 over the same days: $%.0f." % market_line.iloc[-1])

**What drove it.** For each stock you hold now: the dollars you put in, times its return since you bought it.
Green added money, black took it away.

In [ ]:
if record_link.strip() == '':
    print("No Studio 5 fund loaded. Go on to Part 2.")
else:
    # the newest version of my fund that has started counting
    for entry in my_record['entries']:
        if entry['counts_from'] is not None:
            newest = entry
    old_mix = newest['portfolios']['with_rules']
    bought_day = pd.Timestamp(newest['counts_from'])
    money_then = float(my_line.loc[bought_day])

    # a stock I hold that we have no prices for yet: download it
    for t in old_mix:
        if t not in prices.columns:
            one = yf.download(t, start='2021-09-01', interval='1d', auto_adjust=True, progress=False)['Close'].squeeze()
            prices[t] = one.reindex(prices.index).ffill()

    # dollars each stock added = dollars I put in x its return since I bought it
    dollars = {}
    for t in old_mix:
        put_in = money_then * old_mix[t]
        dollars[t] = put_in * (prices.loc[last_close, t] / prices.loc[bought_day, t] - 1)
    bars = pd.Series(dollars).sort_values()

    colors = []
    for t in bars.index:
        if bars[t] >= 0:
            colors.append('tab:green')
        else:
            colors.append('black')
    fig, panel = subplots(figsize=(9, max(3.5, 0.3 * len(bars))))
    panel.barh(bars.index, bars.values, color=colors)
    panel.axvline(0, color='gray', linewidth=1)
    panel.set_title('Dollars each of my stocks added since %s' % bought_day.strftime('%b %d'))
    show()

    print("Helped the most: %s (%+.0f dollars). Hurt the most: %s (%+.0f dollars)." % (bars.index[-1], bars.iloc[-1], bars.index[0], bars.iloc[0]))

### The class

Every student fund is a row, biggest holding on the left. Each box is a stock, colored by its return since Nov 5:
green went up, red went down. The page **Who holds what** on the leaderboard has the same thing with every share:
https://storage.googleapis.com/opim5641-leaderboard-f26/holdings.html

In [ ]:
since_start = 100 * (prices.loc[last_close] / prices.loc[class_start] - 1)        # every stock's return since Nov 5

# one row per fund, one box per holding
widest = 0
for fund in class_funds:
    widest = max(widest, len(fund['weights']))

grid = np.full((len(class_funds), widest), np.nan)
fund_names = []
fig, panel = subplots(figsize=(1.0 * widest + 3, 0.6 * len(class_funds) + 1))
for i in range(len(class_funds)):
    mix = class_funds[i]['weights']
    fund_returns = 100 * (hold(mix, class_start).iloc[-1] / 10000 - 1)
    fund_names.append('%s  %+.1f%%' % (class_funds[i]['name'], fund_returns))
    k = 0
    for t in sorted(mix, key=mix.get, reverse=True):
        grid[i, k] = since_start[t]
        panel.text(k, i, '%s\n%+.0f%%' % (t, since_start[t]), ha='center', va='center', fontsize=7.5)
        k += 1
panel.imshow(grid, cmap='RdYlGn', vmin=-15, vmax=15, aspect='auto')
panel.set_yticks(np.arange(len(class_funds)))
panel.set_yticklabels(fund_names)
panel.set_xticks([])
panel.set_title("Every fund in the class and its return since %s (each fund's newest mix, held from that day)" % class_start.strftime('%b %d'))
show()

crowd_return = 100 * (crowd_line.iloc[-1] / 10000 - 1)
spy_return = 100 * (spy_line.iloc[-1] / 10000 - 1)
print("Put the same dollars in every one of these funds and you have the class crowd fund: %+.2f%% since %s." % (crowd_return, class_start.strftime('%b %d')))
print("The S&P 500 over the same days: %+.2f%%." % spy_return)

# Part 2 - The quilt

The 100 biggest companies, one row for each sector, the biggest company on the left. Every company sits in the **same
square in all three pictures**, so you can follow one stock across three stretches of time:

* **since Nov 5** (the last two weeks),
* **2026 so far**,
* **the last 3 years**.

Green went up, red went down. The numbers are percent returns.

First the numbers behind it. `menu` has one row per company. **Average month** is the number Ms. Womack's model
uses: a stock's average monthly return, here over the last 5 years and over the last 1 year.

In [ ]:
# the three stretches of time
year_start = prices.loc['2026-01-01':].index[0]
three_years_ago = prices.loc[last_close - pd.DateOffset(years=3):].index[0]

# month-end prices and monthly returns (complete months only: this month is not over yet)
month_end = prices.groupby(prices.index.to_period('M')).last()
month_end = month_end[month_end.index < last_close.to_period('M')]
monthly = month_end.pct_change(fill_method=None).iloc[1:]

menu = sp500.loc[biggest, ['company', 'sector']].copy()
menu['since Nov 5 (%)'] = 0.0
menu['2026 so far (%)'] = 0.0
menu['last 3 years (%)'] = 0.0
menu['average month, 5 years (%)'] = 0.0
menu['average month, 1 year (%)'] = 0.0
for t in biggest:
    menu.loc[t, 'since Nov 5 (%)'] = round(100 * (prices.loc[last_close, t] / prices.loc[class_start, t] - 1), 1)
    menu.loc[t, '2026 so far (%)'] = round(100 * (prices.loc[last_close, t] / prices.loc[year_start, t] - 1), 1)
    menu.loc[t, 'last 3 years (%)'] = round(100 * (prices.loc[last_close, t] / prices.loc[three_years_ago, t] - 1), 1)
    menu.loc[t, 'average month, 5 years (%)'] = round(100 * monthly[t].tail(60).mean(), 2)
    menu.loc[t, 'average month, 1 year (%)'] = round(100 * monthly[t].tail(12).mean(), 2)

print("Since Nov 5:   %d of the 100 are up" % (menu['since Nov 5 (%)'] > 0).sum())
print("2026 so far:   %d of the 100 are up" % (menu['2026 so far (%)'] > 0).sum())
print("Last 3 years:  %d of the 100 are up" % (menu['last 3 years (%)'] > 0).sum())

In [ ]:
# the same layout three times: one row per sector, companies from biggest to smallest
sectors = sorted(menu['sector'].unique())
widest = menu['sector'].value_counts().max()

# each picture has its own color scale: (column, the return that gets the darkest color)
pictures = [('since Nov 5 (%)', 15), ('2026 so far (%)', 60), ('last 3 years (%)', 150)]

fig, panels = subplots(3, 1, figsize=(0.62 * widest + 3, 3 * (0.62 * len(sectors) + 1)))
for p in range(3):
    column = pictures[p][0]
    darkest = pictures[p][1]
    grid = np.full((len(sectors), widest), np.nan)
    for i in range(len(sectors)):
        row = menu[menu['sector'] == sectors[i]]          # already in order of size
        k = 0
        for t in row.index:
            grid[i, k] = row.loc[t, column]
            if row.loc[t, column] == row.loc[t, column]:
                label = '%s\n%+.0f' % (t, row.loc[t, column])
            else:
                label = '%s\nnew' % t          # a company that has not traded for the whole stretch
            panels[p].text(k, i, label, ha='center', va='center', fontsize=6.5)
            k += 1
    panels[p].imshow(grid, cmap='RdYlGn', vmin=-darkest, vmax=darkest, aspect='auto')
    panels[p].set_yticks(np.arange(len(sectors)))
    panels[p].set_yticklabels(sectors)
    panels[p].set_xticks([])
    panels[p].set_title('The 100 biggest companies: %s' % column)
tight_layout()
show()

**Read the quilt.** Find a square that is green in all three pictures. Find one that is green over three years
and red since Nov 5. Find NVIDIA (`NVDA`, Technology row).

### The same 100 as a table

Change `sort_by` to any column name and run the cell again. `how_many` is the number of rows to show.

In [ ]:
sort_by = 'since Nov 5 (%)'       # or '2026 so far (%)', 'last 3 years (%)', 'average month, 5 years (%)', 'average month, 1 year (%)'
how_many = 15

look = menu.sort_values(sort_by, ascending=False)
top_and_bottom = pd.concat([look.head(how_many), look.tail(how_many)])

print("The %d best and the %d worst of the 100, by: %s" % (how_many, how_many, sort_by))
top_and_bottom.style.background_gradient(cmap='RdYlGn', subset=['since Nov 5 (%)'], vmin=-15, vmax=15) \
    .background_gradient(cmap='RdYlGn', subset=['2026 so far (%)'], vmin=-60, vmax=60) \
    .background_gradient(cmap='RdYlGn', subset=['last 3 years (%)'], vmin=-150, vmax=150) \
    .format(precision=1)

# Part 3 - My picks

Choose **10 to 25** stocks. The model decides how much of each to buy, and it may buy none of some, so pick a few more
than ten.

There is more than one way to choose, and good managers disagree about which is best:

* **Momentum:** buy what has been going up.
* **Bargains:** buy good companies that have fallen lately.
* **Steady:** buy companies that rarely have a bad year.
* **A view:** buy the companies or sectors you believe in.

To get you started, the next cell prints three lists you can copy.

In [ ]:
momentum = list(menu.sort_values('since Nov 5 (%)', ascending=False).head(15).index)
long_run = list(menu.sort_values('last 3 years (%)', ascending=False).head(15).index)

# bargains: up over three years, down since Nov 5 (the biggest recent falls first)
fallen = menu[(menu['last 3 years (%)'] > 0) & (menu['since Nov 5 (%)'] < 0)]
bargains = list(fallen.sort_values('since Nov 5 (%)').head(15).index)

print("Momentum (best since Nov 5):")
print(momentum)
print()
print("Long-run winners (best over 3 years):")
print(long_run)
print()
print("Bargains (up over 3 years, down since Nov 5):")
print(bargains)

### My picks and my strategy

Edit the list. Tickers go in quotes, with commas between them. Then write your strategy in a few words: it goes on
the leaderboard next to your fund.

In [ ]:
my_picks = ['NVDA', 'AVGO', 'AMD', 'LLY', 'JNJ', 'UNH', 'XOM', 'CVX', 'JPM', 'GS',
            'COST', 'WMT', 'KO', 'CAT', 'GE', 'TSLA']

my_strategy = "a mix: chip winners, plus steady names in every sector"      # a few words, in my own words

In [ ]:
# a pick that is not on the menu: look it up in the S&P 500 table and download its prices
for t in my_picks:
    if t not in sp500.index:
        print(t, "is not in the S&P 500 table. Check the ticker.")
    elif t not in prices.columns:
        one = yf.download(t, start='2021-09-01', interval='1d', auto_adjust=True, progress=False)['Close'].squeeze()
        prices[t] = one.reindex(prices.index).ffill()
        month_end = prices.groupby(prices.index.to_period('M')).last()
        month_end = month_end[month_end.index < last_close.to_period('M')]
        monthly = month_end.pct_change(fill_method=None).iloc[1:]

# my picks, with their sectors and the same numbers as the menu
rows = []
for t in my_picks:
    row = {}
    row['stock'] = t
    row['company'] = sp500.loc[t, 'company']
    row['sector'] = sp500.loc[t, 'sector']
    row['since Nov 5 (%)'] = round(100 * (prices.loc[last_close, t] / prices.loc[class_start, t] - 1), 1)
    row['last 3 years (%)'] = round(100 * (prices.loc[last_close, t] / prices.loc[three_years_ago, t] - 1), 1)
    row['average month, 5 years (%)'] = round(100 * monthly[t].tail(60).mean(), 2)
    row['average month, 1 year (%)'] = round(100 * monthly[t].tail(12).mean(), 2)
    rows.append(row)
picks = pd.DataFrame(rows).set_index('stock')

print("My strategy:", my_strategy)
print("I picked", len(picks), "stocks in", picks['sector'].nunique(), "sectors.")
if len(picks) < 10:
    print("A final fund holds at least 10 stocks, so pick at least 10 (a few more is better).")
picks

**Look at the last two columns.** A stock's average month over 5 years and over 1 year can be very different.
The model only sees the one you tell it to read (`fit_years`, in the next part).

# Part 4 - Solve

Ms. Womack's model with your business rules: the highest average return for the risk you allow.

$$
\begin{aligned}
\max \quad & \sum_k \bar r_k \, x_k && \text{(average return)} \\
\text{subject to} \quad & \sum_k x_k = 1 && \text{(invest all of the money)} \\
& \sum_i \sum_j x_i \, \sigma_{ij} \, x_j \le \text{risk limit} && \text{(Ms. Womack's risk)} \\
& \text{floor} \cdot y_k \le x_k \le \text{cap} \cdot y_k, \quad y_k \in \{0, 1\} && \text{(hold it properly, or not at all)} \\
& \sum_k y_k \ge 10 && \text{(at least 10 stocks)}
\end{aligned}
$$

The next cell is the model as a function: the Studio 5 model with the rules above. It reads your picks' ingredients
(`df_return`, `df_cov`) and your `settings`, and returns the mix it chose. Run it once.

In [ ]:
def solve_portfolio(risk_limit, use_rules):
    model = ConcreteModel()

    # how much of the money goes in each stock
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    # objective: the average return of the portfolio
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()

    # invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # the risk: the weighted covariance matrix, summed
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    if use_rules:
        # the yes/no decision: do we hold stock s?
        model.y = Var(stocks, domain=Binary)
        held = 0
        for s in stocks:
            model.y[s] = 1
            model.constraints.add(model.x[s] <= settings['max_share'] * model.y[s])      # the cap
            model.constraints.add(model.x[s] >= settings['min_share'] * model.y[s])      # the floor
            held += model.y[s]
        model.constraints.add(held >= settings['min_stocks'])                            # at least this many stocks
        results = SolverFactory('bonmin', executable=bonmin_path).solve(model)
    else:
        results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    if results.solver.termination_condition != TerminationCondition.optimal:
        return None

    # keep the stocks the model actually holds (more than 0.1% of the money)
    mix = {}
    for s in stocks:
        if model.x[s]() > 0.001:
            mix[s] = round(model.x[s](), 6)
    return {'mix': mix, 'return': model.portfolio_return(), 'risk': value(risk_expr)}


print("solve_portfolio is ready")

### My settings

These are yours to change. The cap can be anything up to 25%, and the fund must hold at least 10 stocks.

* `fit_years`: how many years of monthly returns the model reads. A short history reacts to recent news; a long one
  is steadier.
* `risk_times`: how much risk you allow. `1.0` is the risk of splitting your money evenly across your picks. You will
  see where that puts you on the efficient frontier in a moment.

In [ ]:
settings = {
    'fit_years': 5,        # years of monthly returns the model reads: 5, 3 or 1
    'max_share': 0.25,     # the cap: at most this much in one stock (0.25 or less)
    'min_share': 0.02,     # the floor: if I hold a stock, at least this much
    'min_stocks': 10,      # hold at least this many stocks (10 or more)
    'risk_times': 1.0,     # 1.0 = as risky as an even split of my picks; 1.3 = 30% more risk; 0.8 = 20% less
}

### Step 1 - The model's two ingredients

Each pick's **average month** (`df_return`) and the **covariance matrix** (`df_cov`), from the months you told the
model to read. Then the yardstick from Studio 5: the risk of an **even split** of your picks.

In [ ]:
# the final-fund rules: a cap of 25% at most, and at least 10 stocks
if settings['max_share'] > 0.25:
    settings['max_share'] = 0.25
    print("The cap is 25% at most, so max_share is set to 0.25.")
if settings['min_stocks'] < 10:
    settings['min_stocks'] = 10
    print("A final fund holds at least 10 stocks, so min_stocks is set to 10.")

# monthly returns for my picks, the last fit_years years
returns = monthly[list(picks.index)].tail(12 * settings['fit_years'])

# a pick that has not traded for that whole history cannot be used: drop it, and say so
for t in returns.columns:
    if returns[t].isna().sum() > 0:
        print(t, "has not traded for all", settings['fit_years'], "years, so the model cannot use it. Try a shorter fit_years.")
returns = returns.dropna(axis=1)
stocks = list(returns.columns)

# the two ingredients
df_return = returns.mean()
df_cov = returns.cov()

# the yardstick: the risk and the average month of an even split of my picks
even_risk = 0
even_return = 0
for i in stocks:
    even_return += df_return[i] / len(stocks)
    for j in stocks:
        even_risk += df_cov.loc[i, j] / len(stocks) ** 2

print("The model reads %d months (%s to %s) for %d picks." % (len(returns), returns.index[0], returns.index[-1], len(stocks)))
print("An even split of my picks: average month %.2f%%, risk %.4f." % (100 * even_return, even_risk))

### Step 2 - The efficient frontier

Solve the model at eight levels of risk, from 60% of the even split's risk up to twice as much. Each solve is a whole
portfolio. We do it twice: with your rules (green) and with no rules at all (gray).

This takes about 20 seconds: the green line needs Bonmin, because of the yes/no variables.

In [ ]:
levels = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]        # risk, as a multiple of the even split's risk

frontier = {}          # level -> the fund with my rules at that level of risk (None if nothing fits)
free_frontier = {}     # level -> the same with no rules
for level in levels:
    frontier[level] = solve_portfolio(even_risk * level, use_rules=True)
    free_frontier[level] = solve_portfolio(even_risk * level, use_rules=False)

# one row per level: the average month each one reaches
rows = []
for level in levels:
    row = {'risk (x even split)': level}
    if frontier[level] is not None:
        row['with my rules: average month (%)'] = round(100 * frontier[level]['return'], 2)
        row['stocks held'] = len(frontier[level]['mix'])
    if free_frontier[level] is not None:
        row['no rules: average month (%)'] = round(100 * free_frontier[level]['return'], 2)
        row['stocks held, no rules'] = len(free_frontier[level]['mix'])
    rows.append(row)
frontier_table = pd.DataFrame(rows).set_index('risk (x even split)')
frontier_table

**The two pictures.** Left: the frontier. Further right is more risk, higher is more average return. The gap
between the gray line and the green line is what your rules cost on paper. The black dot is the even split, and the
dashed line is the risk you chose (`risk_times`).

Right: the **allocation** behind the green line. Each color is a stock, and each column is one portfolio. Watch how
the money moves as you allow more risk.

In [ ]:
fig, (left, right) = subplots(1, 2, figsize=(16, 5.5))

# left: the two frontiers (only the levels that solved)
ruled_x = []
ruled_y = []
free_x = []
free_y = []
for level in levels:
    if frontier[level] is not None:
        ruled_x.append(level)
        ruled_y.append(100 * frontier[level]['return'])
    if free_frontier[level] is not None:
        free_x.append(level)
        free_y.append(100 * free_frontier[level]['return'])
left.plot(free_x, free_y, color='gray', marker='o', linewidth=2, label='no rules')
left.plot(ruled_x, ruled_y, color='tab:green', marker='o', linewidth=3, label='with my rules')
left.scatter([1.0], [100 * even_return], color='black', s=90, zorder=5, label='even split of my picks')
left.axvline(settings['risk_times'], color='black', linestyle='--', linewidth=1, label='my risk (risk_times)')
left.set_xlabel('risk, as a multiple of the even split')
left.set_ylabel('average month (%)')
left.set_title('The efficient frontier for my picks')
left.legend()

# right: the allocation with my rules, one stacked column per level of risk
bottom = np.zeros(len(ruled_x))
shade = 0
for t in stocks:
    heights = []
    for level in ruled_x:
        heights.append(100 * frontier[level]['mix'].get(t, 0))
    if max(heights) > 0:
        right.bar(range(len(ruled_x)), heights, bottom=bottom, label=t, width=0.85, color=cm.tab20(shade % 20))
        bottom = bottom + np.array(heights)
        shade += 1
right.set_xticks(range(len(ruled_x)))
right.set_xticklabels(ruled_x)
right.set_xlabel('risk, as a multiple of the even split')
right.set_ylabel('share of the fund (%)')
right.set_title('The allocation behind the green line')
right.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.01, 1), loc='upper left')
tight_layout()
show()

### Step 3 - My fund

Now one solve at the risk you chose: `risk_times` times the even split's risk. This is the fund you will send.

In [ ]:
risk_limit = even_risk * settings['risk_times']

fund = solve_portfolio(risk_limit, use_rules=True)       # my fund
twin = solve_portfolio(risk_limit, use_rules=False)      # its no-rules twin (the leaderboard tracks both)

if fund is None or twin is None:
    print("No portfolio fits at this level of risk. Raise risk_times or add picks, then run Part 4 again from Step 1.")
else:
    my_mix = fund['mix']

    # my fund as a table, biggest share first
    rows = []
    for t in sorted(my_mix, key=my_mix.get, reverse=True):
        row = {}
        row['stock'] = t
        row['company'] = picks.loc[t, 'company']
        row['sector'] = picks.loc[t, 'sector']
        row['share of my fund (%)'] = round(100 * my_mix[t], 1)
        row['average month the model read (%)'] = round(100 * df_return[t], 2)
        rows.append(row)
    my_fund = pd.DataFrame(rows).set_index('stock')

    # the picks the model left out
    left_out = []
    for t in stocks:
        if t not in my_mix:
            left_out.append(t)

    print("My fund holds %d stocks. Biggest share: %.0f%%." % (len(my_mix), 100 * max(my_mix.values())))
    print("Average month: %.2f%% (an even split of my picks: %.2f%%)." % (100 * fund['return'], 100 * even_return))
    print("Picks the model did not buy:", left_out)
    print("With no rules the model would hold only", len(twin['mix']), "stocks.")
    display(my_fund)

**How would this mix have done?** \$10,000 in today's mix since the first trading day of 2026, next to the S&P 500.
This is hindsight: you picked these stocks already knowing how 2026 went. The red line is Nov 5.

In [ ]:
this_year = hold(my_mix, year_start)
market_year = hold({'SPY': 1.0}, year_start)

fig, panel = subplots(figsize=(11, 4.5))
panel.plot(market_year.index, market_year.values, color='black', linewidth=2, label='S&P 500')
panel.plot(this_year.index, this_year.values, color='tab:green', linewidth=2.5, label='my final fund (this mix)')
panel.axvline(class_start, color='tab:red', linestyle='--', linewidth=1.5)
panel.set_title('$10,000 in this mix since the start of 2026 (hindsight)')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
panel.legend()
show()

print("2026 so far: this mix turned $10,000 into $%.0f. The S&P 500: $%.0f." % (this_year.iloc[-1], market_year.iloc[-1]))
print("Since %s: this mix %+.1f%%, the S&P 500 %+.1f%%." % (
    class_start.strftime('%b %d'), 100 * (this_year.iloc[-1] / this_year.loc[class_start] - 1), spy_return))

**Try things.** Change your picks (Part 3) or your settings, click the **Step 1** cell above, and choose
Runtime, **Run cell and below**. About 30 seconds.

* Set `fit_years` to 1. Which stocks does the model drop, and which does it add?
* Lower the cap to 0.15. How much average return does that cost?
* Swap your picks for the `momentum` list, then for the `bargains` list. Which fund would you rather own for the next
  two weeks, and why?

When you have the fund you want, go on.

# Part 5 - The news (optional)

This cell saves one text file, `my_final_fund_news.txt`: your fund, the numbers for each stock, and up to eight recent
headlines for each one with the link to the story. In Colab it downloads automatically.

Attach the file in Claude (or another AI assistant) and paste the prompt the cell prints. Check what it tells you
against the headlines: you can click every link.

In [ ]:
lines = []
lines.append("MY FINAL FUND. Strategy: %s" % my_strategy)
lines.append("Prices through the %s close. Returns 'since Nov 5' are from the %s close." % (last_close.strftime('%b %d, %Y'), class_start.strftime('%b %d')))
lines.append("")

since = class_start.strftime('%Y-%m-%d')
for t in my_fund.index:
    name = my_fund.loc[t, 'company']
    lines.append("%s (%s), %s: %.1f%% of my fund. Since Nov 5: %+.1f%%. Last 3 years: %+.1f%%." % (
        name, t, my_fund.loc[t, 'sector'], my_fund.loc[t, 'share of my fund (%)'],
        picks.loc[t, 'since Nov 5 (%)'], picks.loc[t, 'last 3 years (%)']))

    # up to eight headlines about this company since Nov 5
    search = '%s stock after:%s' % (name, since)
    link = 'https://news.google.com/rss/search?q=' + requests.utils.quote(search) + '&hl=en-US&gl=US&ceid=US:en'
    try:
        feed = ET.fromstring(requests.get(link, timeout=20).content)
        shown = 0
        for item in feed.iter('item'):
            lines.append("   - (%s) %s" % (item.findtext('pubDate')[5:16], item.findtext('title')))
            lines.append("     %s" % item.findtext('link'))
            shown += 1
            if shown == 8:
                break
    except Exception as problem:
        lines.append("   (no headlines came back)")
    lines.append("")

open('my_final_fund_news.txt', 'w', encoding='utf-8').write("\n".join(lines))
print("Saved my_final_fund_news.txt: headlines for", len(my_fund), "stocks.")
print()
print("A prompt to paste into Claude with the file attached:")
print()
print("   This file lists the stocks in my class investment fund, how each has done, and recent headlines for each.")
print("   For every stock, tell me in two sentences what has been happening, using the headlines.")
print("   Then name the two holdings with the most risk over the next two weeks, and say why.")

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('my_final_fund_news.txt')

# Part 6 - Name it and send it

Your final fund goes on the leaderboard under your NetID. If you already have a fund there, this becomes its newest
version, and the board keeps the history. The name you type gets **FINAL:** in front of it, so everyone can see which
funds are final.

Fill in the four lines. Dr. Dave writes the class code on the board.

In [ ]:
netid = ""             # my UConn NetID, for example abc12345
class_code = ""        # Dr. Dave writes it on the board
final_name = ""        # my fund's name, up to 23 letters, for example "Chips and Steady"
my_reason = ""         # one or two sentences: why these stocks and these settings

#### What the leaderboard will save

Your fund (the model's mix with your rules), its no-rules twin, your settings and your words.

In [ ]:
# each pick's sector
sectors = {}
for t in stocks:
    sectors[t] = picks.loc[t, 'sector']

entry = {
    # who and what
    'netid': netid.strip().lower(),
    'nickname': ('FINAL: ' + final_name.strip())[:30],
    'theme': my_strategy.strip()[:80],
    'reason': ("Studio 6 final fund. " + my_reason.strip())[:200],
    'class_code': class_code.strip(),
    # the two portfolios: my fund, and the same picks with no rules
    'portfolios': {'with_rules': my_mix, 'no_rules': twin['mix']},
    # my settings
    'risk_limit': float(risk_limit),
    'rules': {'min_share': settings['min_share'], 'max_share': settings['max_share'], 'must_have_sectors': [], 'max_stocks': None},
    'extra_rules': {'sector_cap': None, 'min_stocks': settings['min_stocks'], 'must_hold': [], 'never_hold': [], 'either_or': [], 'if_then': []},
    'data_window': [str(returns.index[0]), str(returns.index[-1])],
    # labels the leaderboard uses
    'stock_theme': 'final fund',
    'sectors': sectors,
    'decision': 'final',
    'notebook': 'studio6',
}

print("Fund name on the board:", entry['nickname'])
print("Strategy:              ", entry['theme'])
print("Stocks held:           ", len(my_mix))

#### Check it

Is anything missing, and does the fund keep the two final-fund rules?

In [ ]:
missing = []
if entry['netid'] == '':
    missing.append('netid')
if entry['class_code'] == '':
    missing.append('class_code')
if final_name.strip() == '':
    missing.append('final_name')
if len(my_reason.strip()) < 30:
    missing.append('my_reason (one or two full sentences)')
if len(my_mix) < 10:
    missing.append('at least 10 stocks in the fund (it holds %d)' % len(my_mix))
if max(my_mix.values()) > 0.2505:
    missing.append('at most 25% in one stock')

if len(missing) == 0:
    print("Ready to send.")
else:
    print("Not ready. Still needed:", ", ".join(missing))

#### Send

Colab asks you to sign in with Google (a pop-up: pick your account and click Allow). The leaderboard answers with a
receipt.

In [ ]:
if len(missing) > 0:
    print("Not sent yet. Still needed:", ", ".join(missing))
elif 'google.colab' not in sys.modules:
    print("Not sent: run this notebook in Google Colab to sign in and send.")
else:
    # sign in with Google, then send the entry to the leaderboard
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    answer = requests.post('https://us-central1-opim5641-portfolio-f26.cloudfunctions.net/submit-portfolio',
                           data=json.dumps(entry, default=str), timeout=60,
                           headers={'Authorization': 'Bearer ' + credentials.token, 'Content-Type': 'application/json'})
    receipt = answer.json()
    if receipt.get('ok'):
        print("Saved. This is what the leaderboard has:")
        print("  fund:        ", receipt['nickname'])
        print("  NetID:       ", receipt['netid'], "(signed in as " + receipt['google_email'] + ")")
        print("  stocks held: ", receipt['holdings']['with_rules'])
        print("  It counts from the next market close (4 PM Eastern on a weekday).")
        print("  My fund's page:", receipt.get('fund_page'))
        print("  The leaderboard: https://storage.googleapis.com/opim5641-leaderboard-f26/index.html")
    else:
        print("Not accepted:", receipt.get('problem'))

# After tonight

Your final fund runs until the market closes on **Thursday, Dec 3**. The fund worth the most at that close wins the
class race, and Dr. Dave writes its manager a LinkedIn recommendation.

You can change your fund before then: open this notebook, Run all, change your picks or settings, and send again.
Each send counts from the next market close and is saved with its date and your reason.

**Your final project** is a short report about this fund, due **Monday, Dec 7**. Dr. Dave will post the details.

### Before you leave tonight (Studio 6 credit)

Double-click and answer, two or three sentences each.

**1. My strategy: which stocks did I pick, and why those?**

*My answer:*

**2. My settings: what did I choose for the history, the cap and the risk, and what did the model do with my picks?**

*My answer:*

Then save this notebook (File, Download, Download .ipynb) and hand it in on HuskyCT.